[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/03_surrogate_dataset.ipynb)

# 03 — Surrogate Dataset

**Purpose.** Sample tilt configurations across scenarios, ray-trace each one,
and record the reference radio map. The result is `D_sur = {(x, tilt, R)}` —
PROJECT.md section 11.2 and section 16, Phase 4.

The target is the **radio map**, not the five KPIs (Decision 6). The KPIs are
derived from `R` by `src/kpi/` whenever they are needed, so the dataset does not
bake in the current thresholds and one KPI implementation serves both the
simulator and the model.

**This is the expensive notebook.** Every row costs one ray-tracing solve, and
that budget is the binding constraint on everything downstream. The sampling
design matters more than the surrogate architecture: a well-spread few hundred
configurations will beat a poorly-spread few thousand.

**Inputs.** The scenarios from notebooks 00–02.

**Outputs.** `data/processed/surrogate_dataset.parquet`, split at scenario level.

**Requires** `uv sync --extra rt`. Run it somewhere it can finish.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/data.yaml` resolve the same way
they do for `task clean:data` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the packages Colab does not ship.
Note that `data/` and `models/` are DVC-tracked and therefore *not* part of the
clone — a fresh runtime has neither. See the Drive cell below.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook except the COLAB_PACKAGES line below, which names
# the extras this particular notebook needs. Forked the repository? Change these
# three values and the badge URL at the top of this notebook.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). Colab already ships numpy, pandas, pyarrow,
# scikit-learn, joblib, matplotlib and seaborn, so only these are installed —
# which keeps the bootstrap fast and avoids a "restart runtime" prompt.
COLAB_PACKAGES = [("hydra", "hydra-core"), ("sionna_rt", "sionna-rt")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Empty unless
# the Drive cell below fills it in, so local runs are unaffected.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ and models/ are DVC-tracked, so they are not in the Git clone and a
# fresh Colab runtime has neither. Mount Drive and point the config at it —
# Drive also survives a runtime reset, which /content does not.
#
# The scene is the large one: data/external/simulation_map/ holds 3,753 meshes,
# so keep it on Drive rather than re-downloading it per session.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/band-tilt/data"
# CONFIG_OVERRIDES += [
#     f"data.mdt_path={DATA_ROOT}/raw/measurement_data.csv",
#     f"data.cell_config_path={DATA_ROOT}/external/gcell_conf.csv",
#     f"data.scene_file={DATA_ROOT}/external/simulation_map/scene.xml",
#     f"data.train_path={DATA_ROOT}/processed/mdt_train.parquet",
#     f"data.test_path={DATA_ROOT}/processed/mdt_test.parquet",
# ]

## 1. Setup

Compose the config, seed everything, and import from `src/`. Every notebook
starts the same way so that a cell copied between notebooks behaves identically.

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()  # matplotlib/seaborn styling for report-ready figures

pd.set_option("display.max_columns", 50)
cfg

## 2. Cell-band table and search space

The same table and the same bounds both optimizers will use. Building the design
against a different space from the one that will be searched is the quiet way to
make the surrogate useless exactly where it is needed.

In [ ]:
from src.data.load import load_cell_config, load_processed
from src.optim.space import TiltSpace
from src.radio import cell_band, sampling

table = cell_band.build_table(load_cell_config(cfg), cfg)
space = TiltSpace(table, cfg)
print(f"search space: {space.n_dims} dimensions")
print(f"budget: {cfg.radio.sampling.n_samples} configurations, strategy {cfg.radio.sampling.strategy}")

## 3. Scenarios in the dataset

A sample is a `(scenario, tilt)` pair, and the split falls between **scenarios**
(PROJECT.md section 12.3, Decision 8). Each scenario is one environment plus one
UE mobility realisation, perturbed from the base configuration per section 12.

The budget therefore has two dimensions, and they trade off. Many tilts in one
scenario teaches the surrogate the tilt response and nothing about
generalisation; many scenarios with few tilts each teaches the reverse. Decide
the split of the budget deliberately and record it — it determines what the
held-out error in notebook 04 is actually measuring.

In [ ]:
# TODO(1): enumerate the scenarios available, with their perturbation parameters
# TODO(2): assign each to train / val / test — WHOLE scenarios, never split one
# TODO(3): report configurations per scenario, and the total solve budget
# TODO(4): assert no scenario_id appears in more than one partition
raise NotImplementedError("scenario enumeration — needs scenario_id from notebook 00")

## 4. Sample tilt configurations

A low-discrepancy design, not uniform draws. At the sample counts affordable
here independent uniform sampling leaves large regions of a high-dimensional
space untouched while clustering elsewhere.

The baseline configuration is included by construction — it is the reference
every claim is measured against, and a surrogate inaccurate exactly there would
undermine every comparison.

In [ ]:
tilts = sampling.sample_configurations(table, cfg)
sampling.assert_within_bounds(tilts, table)

baseline_present = np.isclose(tilts, space.baseline()).all(axis=1).any()
assert baseline_present, "the baseline configuration must be in the design"
print(f"{tilts.shape[0]} configurations x {tilts.shape[1]} cell-bands")

## 5. Inspect the design before spending compute

Cheap to look at, expensive to get wrong. Check that each dimension is covered
across its full range and that the design is not accidentally correlated across
cell-bands — a design where every cell tilts together teaches the surrogate
nothing about coordination, which is the entire point of the study.

In [ ]:
coverage_df = pd.DataFrame(tilts, columns=table["gcell_id"] + "|" + table["band"])
print(coverage_df.describe().T[["min", "max", "mean", "std"]].head(10))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(coverage_df.corr(), center=0, ax=ax, cbar_kws={"label": "correlation"})
ax.set_title("cross-dimension correlation (should be near zero)")

## 6. Generate the reference radio maps

The long run. Two properties the implementation must have:

- **Resumable.** A run that dies at sample 400 of 500 and cannot restart has
  thrown away days of compute.
- **Scene loaded once per scenario.** Rebuilding it per sample dominates the
  runtime.

Each row stores the full RSRP tensor for its configuration, indexed
`(cell_band, grid_cell)` in table order. That is a lot more per row than five
KPI numbers — plan the storage format before starting, not after.

In [ ]:
from src.data.ue_density import ue_density
from src.surrogate import dataset

# UE density is per SCENARIO, not global: each scenario has its own mobility
# realisation, so it is constant across that scenario's configurations only.
rho = ue_density(load_processed(cfg, "train"), cfg)
d_sur = dataset.build(cfg)
print(f"{len(d_sur)} (scenario, tilt) samples")

## 7. Dataset audit

Before training anything, check that the labels span a useful range. If every
configuration produces roughly the same KPIs, the tilt bounds are too narrow to
matter and there is nothing for an optimizer to find — which is a finding, and a
much cheaper one to discover here than after two optimization studies.

The KPIs below are **derived** from the stored maps by `src/kpi/`; they are an
audit of the dataset, not its target.

In [ ]:
kpi_cols = list(cfg.kpi.order)
derived = dataset.derive_kpis(d_sur, rho, table, cfg)
print(derived[kpi_cols].describe().T)

fig, axes = plt.subplots(1, len(kpi_cols), figsize=(16, 3))
for ax, col in zip(axes, kpi_cols, strict=True):
    sns.histplot(derived[col], bins=30, ax=ax)
    ax.set_title(col, fontsize=9)
plt.tight_layout()

# TODO: how much better is the best sampled configuration than the baseline?
# That gap is the headroom any optimizer can possibly claim.

## 8. Persist with provenance

The metadata is what makes the dataset extendable. The ray-tracing settings and
the grid geometry *define* the labels, so a dataset whose provenance is unknown
cannot be added to later — the new rows would come from a different function.

`scenario_id` is part of the provenance, not an afterthought: without it the
scenario-level split cannot be reconstructed and the held-out error becomes
unverifiable.

In [ ]:
provenance = {
    "cell_band_order": (table["gcell_id"] + "|" + table["band"]).tolist(),
    "grid": dict(cfg.radio.grid),
    "ray_tracing": dict(cfg.radio.ray_tracing),
    "sampling": dict(cfg.radio.sampling),
    "target": dict(cfg.surrogate.target),
    "scenario_ids": sorted(d_sur["scenario_id"].unique().tolist()),
    "split_on": cfg.surrogate.dataset.split_on,
    "seed": cfg.seed,
}
dataset.save(d_sur, cfg.surrogate.dataset.path, metadata=provenance)
print(f"wrote {cfg.surrogate.dataset.path}")

## 9. Handoff checklist

- [ ] Every sampled configuration is within its per-cell-band bounds.
- [ ] The baseline configuration is in the dataset.
- [ ] Every row carries `scenario_id`, and no scenario spans two partitions.
- [ ] The stored target is the radio map in dBm, matching the Sionna-RT
      reference — not a normalised or KPI-reduced quantity.
- [ ] The derived KPI ranges in section 7 are wide enough that optimization has
      headroom.
- [ ] Provenance is stored with the dataset, ray-tracing settings included.
- [ ] The dataset is DVC-tracked.

**Blocking gaps.** `scenario_id` has no producer (notebook 00), the perturbed
scenarios of section 12 have no generator, and `src.surrogate.dataset` still
stores KPI labels rather than maps.